In [1]:
import pandas as pd

sample_df = pd.read_csv('/mnt/rice/default/Workspace/yangdong/ai4research/DATA/shang_data/sample_SR_B3.csv')
sample_df.head()

,Accession,Species information,SR_B1,SR_B2,SR_B3
0,NH001,O. sativa japonica,0.000000,0.083333,0.000000
1,NH002,O. sativa japonica,0.125000,0.000000,0.125000
2,NH003,O. sativa japonica,0.130435,0.166667,0.041667
3,NH005,O. sativa japonica,0.000000,0.041667,0.000000
4,NH006,O. sativa indica,0.041667,0.000000,0.166667


In [5]:
import os
import re

all_samples = sample_df['Accession'].tolist()

# 2. 目录路径定义
dirs = {
    "genome": "/mnt/rice/data/OsGenos/251cellresearch/chromosome_level/genome",
    "annotation": "/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/liftoff_IRGSP_full/annotations",
    "leaf_ck": "/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_ck",
    "leaf_salt": "/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_salt"
}

# 3. 初始化结果字典
sample_files = {sample: [] for sample in all_samples}

# 4. 遍历目录并匹配绝对路径
for condition, dir_path in dirs.items():
    if not os.path.exists(dir_path):
        print(f"警告: 路径不存在 -> {dir_path}")
        continue
        
    for root, _, files in os.walk(dir_path):
        for file in files:
            # 仅匹配 .bw 或 .bigwig 文件（可根据实际后缀调整）
            if file.endswith((".fasta", ".gff3", ".bw", ".bigwig")):
                full_path = os.path.abspath(os.path.join(root, file))
                
                # 遍历样本名称进行正则匹配
                for sample in all_samples:
                    # 只要文件名包含 sample，且 sample 后面跟的是非字母数字字符（如 _、.）或结尾，即可匹配
                    pattern = rf"{sample}(?=[^a-zA-Z0-9]|$)"
                    if re.search(pattern, file):
                        sample_files[sample].append(full_path)

# 打印最终构建的字典
import pprint
pprint.pprint(sample_files)

{nan: [],
 'NH001': ['/mnt/rice/data/OsGenos/251cellresearch/chromosome_level/genome/Oryza_sativa_NH001.fasta',
           '/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/liftoff_IRGSP_full/annotations/NH001.IRGSP.full.liftoff.gff3',
           '/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_ck/NH001__Heibiao__SRR13451351.tmm_cpm_log2_plus1_coverage.bw',
           '/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_salt/NH001__Heibiao__SRR25894121.tmm_cpm_log2_plus1_coverage.bw'],
 'NH002': ['/mnt/rice/data/OsGenos/251cellresearch/chromosome_level/genome/Oryza_sativa_NH002.fasta',
           '/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/liftoff_IRGSP_full/annotations/NH002.IRGSP.full.liftoff.gff3',
           '/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_ck/NH002__Sansuijin__SRR13451349.tm

In [7]:
# 过滤字典，仅保留 len(value) == 4 的元素
sample_files = {k: v for k, v in sample_files.items() if len(v) == 4}

# 查看过滤后的样本数量和列表
print(f"有效样本数: {len(sample_files)}")
# for sample, files in sample_files.items():
#     print(sample, "->", len(files), "个文件")

有效样本数: 202


In [8]:
import os
import pyBigWig
import pandas as pd

# 目标基因
target_gene = "Os05g0572000"

def get_gene_coords_from_gff(gff_path, gene_id):
    """从 GFF3 中精准提取指定基因的 chromosome, start, end"""
    if not os.path.exists(gff_path):
        return None
    with open(gff_path, 'r') as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split('\t')
            if len(parts) < 9 or parts[2] != 'gene':
                continue
            
            attr = parts[8]
            # 解析 ID 或 Name 属性
            for item in attr.split(';'):
                if item.startswith('ID=') or item.startswith('Name='):
                    val = item.split('=')[1]
                    # 匹配完整基因名或去除版本号后缀的基因名
                    if val == gene_id or val.split('.')[0] == gene_id:
                        return {
                            'chrom': parts[0],
                            'start': int(parts[3]) - 1,  # 转为 0-based
                            'end': int(parts[4])
                        }
    return None

def get_mean_coverage(bw_path, chrom, start, end):
    """从 BigWig 提取指定区间的 Mean Signal"""
    if not os.path.exists(bw_path):
        return None
    try:
        bw = pyBigWig.open(bw_path)
        chroms = bw.chroms()
        if chrom not in chroms:
            bw.close()
            return None
        
        chrom_len = chroms[chrom]
        real_start = max(0, min(start, chrom_len))
        real_end = max(0, min(end, chrom_len))
        
        if real_start >= real_end:
            bw.close()
            return None
            
        val = bw.stats(chrom, real_start, real_end, type="mean")[0]
        bw.close()
        return float(val) if val is not None else 0.0
    except Exception:
        return None

# 过滤仅保留 value 长度为 4 的样本
valid_samples = {k: v for k, v in sample_files.items() if len(v) == 4}

records = []
for sample, files in valid_samples.items():
    fasta_path, gff_path, ck_bw, salt_bw = files[0], files[1], files[2], files[3]
    
    # 1. 解析当前样本该基因的坐标
    coords = get_gene_coords_from_gff(gff_path, target_gene)
    if not coords:
        # 未匹配到基因坐标时填 None
        records.append({
            'Sample': sample,
            'Gene_ID': target_gene,
            'Chrom': None,
            'Start': None,
            'End': None,
            'CK_Expr': None,
            'Salt_Expr': None
        })
        continue
    
    chrom, start, end = coords['chrom'], coords['start'], coords['end']
    
    # 2. 提取 CK 和 Salt 信号的均值
    ck_val = get_mean_coverage(ck_bw, chrom, start, end)
    salt_val = get_mean_coverage(salt_bw, chrom, start, end)
    
    records.append({
        'Sample': sample,
        'Gene_ID': target_gene,
        'Chrom': chrom,
        'Start': start,
        'End': end,
        'CK_Expr': round(ck_val, 4) if ck_val is not None else None,
        'Salt_Expr': round(salt_val, 4) if salt_val is not None else None
    })

# 3. 转为 DataFrame
df_gene_expr = pd.DataFrame(records)

print(f"成功提取 {len(df_gene_expr)} 个样本的基因 [{target_gene}] 表达量数据：")
print(df_gene_expr.head(10))

成功提取 202 个样本的基因 [Os05g0572000] 表达量数据：
  Sample       Gene_ID            Chrom     Start       End  CK_Expr  \
0  NH001  Os05g0572000  GWHDRAM00000008  28606007  28607249   0.5303   
1  NH002  Os05g0572000  GWHDRAN00000008  28551186  28552431   0.2843   
2  NH003  Os05g0572000  GWHDRAO00000008  28895151  28896396   0.5016   
3  NH005  Os05g0572000  GWHDRAP00000008  28663306  28664551   0.7236   
4  NH006  Os05g0572000  GWHDRAQ00000008  29189220  29190462   0.2692   
5  NH007  Os05g0572000  GWHDRAR00000008  29451354  29452596   1.1006   
6  NH008  Os05g0572000  GWHDRAS00000008  29698708  29699950   1.0575   
7  NH009  Os05g0572000  GWHDRAT00000008  29646327  29647569   1.1975   
8  NH010  Os05g0572000  GWHDRAU00000008  29655248  29656490   0.6330   
9  NH011  Os05g0572000  GWHDRAV00000008  30627979  30629221   0.5255   

   Salt_Expr  
0     3.1801  
1     1.4622  
2     1.9529  
3     1.4599  
4     1.7729  
5     1.3613  
6     1.4253  
7     1.2802  
8     1.4463  
9     0.7394  


In [9]:
import numpy as np

# 1. 计算 Salt / CK 比值 (添加 1e-5 伪计数防止除以 0)
df_gene_expr['Salt_CK_Ratio'] = (df_gene_expr['Salt_Expr'] + 1e-5) / (df_gene_expr['CK_Expr'] + 1e-5)

# 2. 同时计算 Log2FC，更直观显示上调/下调倍数
df_gene_expr['Log2FC'] = np.log2(df_gene_expr['Salt_CK_Ratio'])

# 3. 按 Salt_CK_Ratio 从大到小降序排序 (NaN 值自动排在最后)
df_sorted = df_gene_expr.sort_values(by='Salt_CK_Ratio', ascending=False).reset_index(drop=True)

# 4. 保留合适的小数位数
df_sorted['Salt_CK_Ratio'] = df_sorted['Salt_CK_Ratio'].round(4)
df_sorted['Log2FC'] = df_sorted['Log2FC'].round(4)

print("=== 按 Salt/CK 表达比值从大到小排序后的前 10 个样本 ===")
print(df_sorted[['Sample', 'Gene_ID', 'CK_Expr', 'Salt_Expr', 'Salt_CK_Ratio', 'Log2FC']].head(10))

# 导出为 CSV 表格
df_sorted.to_csv("Os05g0572000_salt_vs_ck_ratio_sorted.csv", index=False)

=== 按 Salt/CK 表达比值从大到小排序后的前 10 个样本 ===
  Sample       Gene_ID  CK_Expr  Salt_Expr  Salt_CK_Ratio  Log2FC
0  NH062  Os05g0572000   0.0114     2.6511       232.3497  7.8602
1  NH153  Os05g0572000   0.0381     1.3149        34.5030  5.1087
2  NH187  Os05g0572000   0.0756     2.0217        26.7387  4.7409
3  NH247  Os05g0572000   0.0591     1.4882        25.1770  4.6540
4  NH063  Os05g0572000   0.0816     2.0438        25.0436  4.6464
5  NH227  Os05g0572000   0.0837     1.8909        22.5888  4.4975
6  NH198  Os05g0572000   0.1150     2.1623        18.8011  4.2327
7  NH237  Os05g0572000   0.1314     2.2028        16.7629  4.0672
8  NH094  Os05g0572000   0.1322     1.9138        14.4755  3.8555
9  NH069  Os05g0572000   0.1664     2.2142        13.3058  3.7340


In [19]:
bw = pyBigWig.open('/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/RNA_bam/analysis/six_context_tmm_bigwig_links/leaf_ck/NH001__Heibiao__SRR13451351.tmm_cpm_log2_plus1_coverage.bw')
# val = bw.stats(chrom, real_start, real_end, type="mean")[0]

print(bw.chroms())
val = bw.stats('GWHDRAM00000005', 28606007, 28607249, type="mean")
bw.close()
print(val)

{'GWHDRAM00000001': 43500536, 'GWHDRAM00000002': 25342113, 'GWHDRAM00000003': 29250786, 'GWHDRAM00000004': 26730703, 'GWHDRAM00000005': 36082031, 'GWHDRAM00000006': 37695486, 'GWHDRAM00000007': 32875394, 'GWHDRAM00000008': 29996347, 'GWHDRAM00000009': 31415545, 'GWHDRAM00000010': 30304653, 'GWHDRAM00000011': 28705863, 'GWHDRAM00000012': 22560978}
[0.0]


[0.0]